# 🥬 Notebook 09 — Pipeline FoodSeg103 v2 (SAM auto-mask + CLIP ensemble + filtro)

Este notebook itera sobre el **nb08**, atacando los problemas que se detectaron en su evaluación cualitativa.

## Qué arregla respecto al nb08

El nb08 mostró tres síntomas (todos derivados de una misma causa raíz):

| Síntoma (nb08) | Causa raíz | Corrección (nb09) |
|---|---|---|
| SAM segmenta solo los ingredientes que YOLO ve | YOLOv8-COCO conoce 10 alimentos, no los 103 ingredientes de FoodSeg103 | **`SamAutomaticMaskGenerator`**: SAM mismo propone máscaras sobre toda la imagen, sin depender de YOLO |
| CLIP top-1 ~36% — confunde clases visualmente parecidas | Un solo template de prompt no captura la variabilidad léxica | **Prompt ensemble**: promediamos embeddings de 6 templates por clase (best practice del paper original de CLIP) |
| Predicciones catastróficas con kcal disparado (ej. "lamb 366 kcal" sobre una zanahoria) | El pipeline acepta cualquier top-1, sin importar la confianza | **Filtro de confianza**: top-1 < umbral → la máscara queda "indeterminada" y no aporta kcal |

## Lo que NO cambia

- El segmentador sigue siendo **SAM vit_b** (mismo checkpoint, mismo modelo).
- El clasificador sigue siendo **CLIP ViT-B/32 openai** (mismo modelo, distinto encoding del lado del texto).
- El lookup nutricional sigue siendo `data/nutrition_foodseg103.json`.
- El presupuesto sigue siendo **250 g por plato** repartido por área de máscara.

## Estructura

1. Setup (incluye el `SamAutomaticMaskGenerator` además del `SamPredictor`).
2. Lookup nutricional (idéntico al nb08).
3. Funciones core v2: `segment_with_sam_auto`, `classify_with_clip_ensemble`, `analyze_dish_foodseg103_v2`.
4. Pipeline integrado + smoke test.
5. Comparativa **segmentación** (mIoU/Dice): v1 (nb08) vs v2 (nb09).
6. Comparativa **clasificación** (top-1/3/5): CLIP simple vs CLIP ensemble.
7. Comparativa **MAPE calórico** end-to-end: v1 vs v2.
8. Visualización cualitativa lado a lado (GT | v1 | v2) — esperamos ver la mejora a ojo.
9. Resumen + conclusiones para el informe.

> ⚠️ Requiere las mismas dependencias del nb08 (`segment-anything`, `open_clip_torch`) y los mismos checkpoints (`weights/sam_vit_b_01ec64.pth`, `weights/model_v1.pt`).
>
> 💡 **Costo computacional**: el auto-mask-generator es ~3-5× más lento que el modo con prompts. Para 100 imágenes esperar ~10-15 min en MPS. Si querés iterar más rápido, bajar `N_SEG_EVAL`.

## ⚙️ Parte 1 — Setup

Mismo setup del nb08 + un `SamAutomaticMaskGenerator` que es el cambio central. Los parámetros (`points_per_side=24`, thresholds de calidad) están seteados para balancear cobertura y velocidad sobre fotos de comida — más puntos → más máscaras pero más lento.

In [ ]:
import sys
sys.path.append('..')

# Forzar tqdm clásico (texto) en vez de widgets
import tqdm.std, tqdm.notebook, tqdm.auto
tqdm.notebook.tqdm = tqdm.std.tqdm
tqdm.notebook.trange = tqdm.std.trange
tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from tqdm import tqdm
from datasets import load_dataset
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator
import open_clip

from src.config import (
    DATA_DIR, WEIGHTS_DIR, DEVICE, SEED,
    FOODSEG103_CLASSES, FOODSEG103_NUTRITION_PATH,
)

rng = np.random.default_rng(SEED)
print(f'Device: {DEVICE}')

# ── SAM ─────────────────────────────────────────────────────────────────
SAM_CHECKPOINT = WEIGHTS_DIR / 'sam_vit_b_01ec64.pth'
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(DEVICE)
# Para comparar contra el nb08 también mantenemos el SamPredictor (modo con boxes).
sam_predictor = SamPredictor(sam)
# Y el AutomaticMaskGenerator (el cambio central del nb09).
sam_auto = SamAutomaticMaskGenerator(
    sam,
    points_per_side=24,             # grid 24x24 = 576 puntos prompt sobre la imagen
    pred_iou_thresh=0.86,           # filtra máscaras de baja calidad predicha
    stability_score_thresh=0.92,    # filtra máscaras inestables
    min_mask_region_area=500,       # filtra máscaras muy chicas (ruido)
)
print(f'SAM vit_b cargado — modos: SamPredictor (boxes) y SamAutomaticMaskGenerator (auto)')

# ── CLIP ────────────────────────────────────────────────────────────────
CLIP_MODEL_NAME, CLIP_PRETRAINED = 'ViT-B-32', 'openai'
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED, force_quick_gelu=True,
)
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
print(f'CLIP {CLIP_MODEL_NAME} ({CLIP_PRETRAINED}) listo')

# ── YOLO (solo para la rama de comparación v1) ─────────────────────────
yolo = YOLO('yolov8n.pt')

# ── FoodSeg103 ──────────────────────────────────────────────────────────
foodseg = load_dataset('EduardoPacheco/FoodSeg103', split='validation')
print(f'FoodSeg103 validation: {len(foodseg)} imágenes')

# ── Embeddings textuales: SIMPLE (nb08) y ENSEMBLE (nb09) ──────────────
# El paper de CLIP recomienda promediar embeddings de varios templates por clase
# y normalizar el promedio. Eso suele subir top-1 unos 3-5 puntos.
CLIP_TEMPLATE_SINGLE = ['a photo of {name}, a type of food']
CLIP_TEMPLATES_ENSEMBLE = [
    'a photo of {name}, a type of food.',
    'a close-up photo of {name}.',
    'a photo of {name} on a plate.',
    'an image of fresh {name}.',
    'a photo of cooked {name}.',
    '{name} ingredient.',
]

def build_text_features(templates):
    """Promedia embeddings de varios templates por clase, normalizando al final."""
    feats_per_class = []
    with torch.no_grad():
        for cls in FOODSEG103_CLASSES:
            name = cls.strip()
            prompts = [t.format(name=name) for t in templates]
            tokens = clip_tokenizer(prompts).to(DEVICE)
            f = clip_model.encode_text(tokens)         # (n_templates, D)
            f = f / f.norm(dim=-1, keepdim=True)
            f = f.mean(dim=0)                          # promedio
            f = f / f.norm()                           # re-normalizar
            feats_per_class.append(f)
    return torch.stack(feats_per_class)                # (103, D)

text_features_v1 = build_text_features(CLIP_TEMPLATE_SINGLE)
text_features_v2 = build_text_features(CLIP_TEMPLATES_ENSEMBLE)
print(f'Embeddings v1 (single template):   shape {tuple(text_features_v1.shape)}')
print(f'Embeddings v2 (ensemble {len(CLIP_TEMPLATES_ENSEMBLE)} templates): shape {tuple(text_features_v2.shape)}')

## 🥗 Parte 2 — Lookup nutricional

Idéntico al nb08.

In [ ]:
nutrition_db = json.load(open(FOODSEG103_NUTRITION_PATH))
print(f'Lookup nutricional: {len(nutrition_db)}/{len(FOODSEG103_CLASSES)} ingredientes')

def get_ingredient_nutrition(name):
    return nutrition_db.get(name) or nutrition_db.get(name.strip())

PLATE_PORTION_G = 250.0

## 🔧 Parte 3 — Funciones core v2

Tres cambios respecto al nb08:

1. **`segment_with_sam_auto(image, top_n)`**: usa `SamAutomaticMaskGenerator` para generar máscaras sobre toda la imagen, sin necesidad de prompts. Filtra a las `top_n` máscaras más grandes (por área) para evitar que el ruido domine el reparto de porción.
2. **`classify_with_clip(crop, use_ensemble)`**: switch entre los embeddings simple (v1) y ensemble (v2), para poder comparar en la Parte 6.
3. **Filtro de confianza**: el caller decide qué hacer si top-1 < umbral (el pipeline v2 lo marca como 'indeterminado'). Por defecto `MIN_CONF = 0.20`.

In [ ]:
MIN_CONF = 0.20    # CLIP top-1 < MIN_CONF -> ingrediente indeterminado
TOP_N_MASKS = 12   # quedarse con las N máscaras más grandes del auto-generator


def segment_with_sam_auto(image, top_n=TOP_N_MASKS, min_area_frac=0.005):
    """Auto-mask: SAM genera máscaras sobre toda la imagen sin prompts.
    Devuelve las top_n máscaras más grandes (área) que superan min_area_frac."""
    arr = np.array(image.convert('RGB'))
    H, W = arr.shape[:2]
    raw_masks = sam_auto.generate(arr)
    masks = []
    for m in raw_masks:
        if m['area'] >= min_area_frac * H * W:
            masks.append(m['segmentation'].astype(bool))
    masks.sort(key=lambda m: -int(m.sum()))
    return masks[:top_n]


def classify_with_clip(crop_pil, topk=3, use_ensemble=True):
    """Cosine similarity contra los 103 embeddings (simple o ensemble)."""
    tf = text_features_v2 if use_ensemble else text_features_v1
    with torch.no_grad():
        x = clip_preprocess(crop_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
        feat = clip_model.encode_image(x)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        sims = (100.0 * feat @ tf.T).softmax(dim=-1)[0]
    top_vals, top_idx = sims.topk(topk)
    return [(FOODSEG103_CLASSES[i.item()], v.item()) for i, v in zip(top_idx, top_vals)]


def crop_from_mask(image, mask, pad=8):
    ys, xs = np.where(mask)
    if len(xs) == 0: return None
    H, W = mask.shape
    return image.crop((max(0, int(xs.min())-pad), max(0, int(ys.min())-pad),
                       min(W, int(xs.max())+pad), min(H, int(ys.max())+pad)))


# Mantenemos también la rama v1 (SAM + prompts YOLO) para poder comparar end-to-end.
FOOD_COCO = {'pizza','donut','cake','sandwich','hot dog','apple','banana','orange','broccoli','carrot'}
SCALE_REF = {'bowl','cup','plate','dining table','fork','knife','spoon','wine glass','bottle'}
PROMPT_COCO = FOOD_COCO | SCALE_REF

def yolo_prompt_boxes(image, conf=0.25, min_area_frac=0.01):
    res = yolo(image, conf=conf, verbose=False)[0]
    W, H = image.size
    out = []
    for box, cls_id in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = box.astype(int)
        if (x2-x1)*(y2-y1)/(W*H) < min_area_frac: continue
        if res.names[int(cls_id)] in PROMPT_COCO:
            out.append((x1, y1, x2, y2))
    return out

def segment_with_sam_boxes(image, boxes):
    arr = np.array(image.convert('RGB'))
    sam_predictor.set_image(arr)
    masks = []
    for box in boxes:
        m, _, _ = sam_predictor.predict(
            point_coords=None, point_labels=None,
            box=np.array(box[:4], dtype=np.float32), multimask_output=False,
        )
        masks.append(m[0].astype(bool))
    return masks

print('Funciones core v2 listas:')
print('  segment_with_sam_auto(image)        -> [mask]  (auto, sin prompts)')
print('  classify_with_clip(crop, use_ensemble=True)')
print(f'  Filtros: MIN_CONF={MIN_CONF}, TOP_N_MASKS={TOP_N_MASKS}')
print('Funciones v1 retenidas para comparación:')
print('  segment_with_sam_boxes(image, boxes)   yolo_prompt_boxes(image)')

## 🔗 Parte 4 — Pipelines integrados v1 y v2

Definimos dos pipelines en paralelo para poder evaluarlos lado a lado:

- **v1 (nb08)**: YOLO (prompts) → SAM con boxes → CLIP single template, sin filtro.
- **v2 (nb09)**: SAM auto-mask → CLIP ensemble → filtro de confianza.

El smoke test compara las dos predicciones sobre la primera imagen del dataset.

In [ ]:
def analyze_dish_v1(image):
    """Pipeline del nb08: YOLO -> SAM (boxes) -> CLIP simple, sin filtro."""
    image = image.convert('RGB')
    W, H = image.size
    boxes = yolo_prompt_boxes(image) or [(0, 0, W, H)]
    masks = segment_with_sam_boxes(image, boxes)
    areas = [int(m.sum()) for m in masks]
    total_area = sum(areas) or 1
    dets = []
    for box, mask, area in zip(boxes, masks, areas):
        crop = crop_from_mask(image, mask)
        if crop is None or min(crop.size) < 10: continue
        topk = classify_with_clip(crop, topk=3, use_ensemble=False)
        cls, conf = topk[0]
        portion = PLATE_PORTION_G * area / total_area
        nut = get_ingredient_nutrition(cls)
        kcal = nut['calories'] * portion / 100 if nut else None
        dets.append(dict(mask=mask, ingredient=cls, confidence=conf,
                         portion_g=portion, kcal=kcal, status='ok'))
    total_kcal = sum(d['kcal'] for d in dets if d['kcal'] is not None)
    return dict(detections=dets, total_kcal=total_kcal)


def analyze_dish_v2(image, min_conf=MIN_CONF):
    """Pipeline nb09: SAM auto-mask -> CLIP ensemble -> filtro de confianza.
    Si CLIP top-1 < min_conf, la máscara queda 'indeterminada' y NO aporta kcal.
    Las kcal se reparten solo entre las máscaras aceptadas."""
    image = image.convert('RGB')
    masks = segment_with_sam_auto(image)
    if not masks:
        return dict(detections=[], total_kcal=0.0)

    # Primera pasada: clasificar cada máscara y decidir cuáles aceptamos
    accepted = []
    rejected = []
    for mask in masks:
        crop = crop_from_mask(image, mask)
        if crop is None or min(crop.size) < 10:
            continue
        topk = classify_with_clip(crop, topk=3, use_ensemble=True)
        cls, conf = topk[0]
        entry = dict(mask=mask, ingredient=cls, confidence=conf,
                     top3=topk, area_px=int(mask.sum()))
        if conf >= min_conf and get_ingredient_nutrition(cls) is not None:
            accepted.append(entry)
        else:
            entry['status'] = 'indeterminado'
            rejected.append(entry)

    # Segunda pasada: reparto de porción solo entre las aceptadas
    accepted_area = sum(e['area_px'] for e in accepted) or 1
    dets = []
    total_kcal = 0.0
    for e in accepted:
        portion = PLATE_PORTION_G * e['area_px'] / accepted_area
        nut = get_ingredient_nutrition(e['ingredient'])
        kcal = nut['calories'] * portion / 100
        dets.append(dict(mask=e['mask'], ingredient=e['ingredient'],
                         confidence=e['confidence'], portion_g=portion,
                         kcal=kcal, status='ok'))
        total_kcal += kcal
    for e in rejected:
        dets.append(dict(mask=e['mask'], ingredient=e['ingredient'],
                         confidence=e['confidence'], portion_g=0.0,
                         kcal=0.0, status='indeterminado'))
    return dict(detections=dets, total_kcal=total_kcal,
                n_accepted=len(accepted), n_rejected=len(rejected))


# Smoke test sobre la primera imagen del dataset
sample = foodseg[0]
r1 = analyze_dish_v1(sample['image'])
r2 = analyze_dish_v2(sample['image'])
print('=== v1 (SAM+YOLO+CLIP simple) ===')
print(f"  {len(r1['detections'])} detecciones, total {r1['total_kcal']:.0f} kcal")
for d in r1['detections'][:8]:
    print(f"    {d['ingredient']:20s} conf={d['confidence']:.2f}  {d['portion_g']:.0f}g  {d['kcal']:.0f} kcal")
print()
print(f"=== v2 (SAM auto + CLIP ensemble + filtro >= {MIN_CONF}) ===")
print(f"  {r2['n_accepted']} aceptadas + {r2['n_rejected']} indeterminadas, total {r2['total_kcal']:.0f} kcal")
for d in r2['detections'][:12]:
    flag = '✓' if d['status'] == 'ok' else '✗'
    print(f"    {flag} {d['ingredient']:20s} conf={d['confidence']:.2f}  {d['portion_g']:.0f}g  {d['kcal']:.0f} kcal")

## 📊 Parte 5 — Comparativa de segmentación: SAM-prompts vs SAM-auto

Misma evaluación del nb08 pero ahora con 3 columnas:
- **Box-as-mask (nb07)**: unión de los rectángulos de YOLO usados como máscaras crudas.
- **SAM + boxes YOLO (nb08, v1)**: máscaras SAM usando las cajas de YOLO como prompts.
- **SAM auto-mask (nb09, v2)**: máscaras SAM generadas sobre toda la imagen sin prompts.

La expectativa: v2 supera ampliamente a v1 en `mIoU/Dice` porque SAM ya no depende del vocabulario limitado de YOLO.

In [ ]:
N_SEG_EVAL = 50    # auto-mask es lento; 50 imágenes ya da una señal estadística clara

def mask_metrics(pred, gt):
    inter = float((pred & gt).sum())
    p, g = float(pred.sum()), float(gt.sum())
    iou = inter / max(p + g - inter, 1.0)
    dice = 2 * inter / max(p + g, 1.0)
    return iou, dice


seg_idxs = rng.choice(len(foodseg), size=N_SEG_EVAL, replace=False)
box_ious, box_dices = [], []
sam_v1_ious, sam_v1_dices = [], []
sam_v2_ious, sam_v2_dices = [], []

for idx in tqdm(seg_idxs, desc='Box vs SAM-v1 vs SAM-v2'):
    s = foodseg[int(idx)]
    image = s['image'].convert('RGB')
    W, H = image.size
    gt = np.array(s['label'])
    if gt.ndim == 3: gt = gt[..., 0]
    gt_fg = gt > 0

    boxes = yolo_prompt_boxes(image) or [(0, 0, W, H)]

    # Box-as-mask
    box_fg = np.zeros_like(gt_fg)
    for x1, y1, x2, y2 in boxes:
        box_fg[max(0,y1):min(H,y2), max(0,x1):min(W,x2)] = True
    iou_b, dice_b = mask_metrics(box_fg, gt_fg)
    box_ious.append(iou_b); box_dices.append(dice_b)

    # SAM v1 (con boxes YOLO)
    m_v1 = segment_with_sam_boxes(image, boxes)
    sam_v1_fg = np.zeros_like(gt_fg)
    for m in m_v1: sam_v1_fg |= m
    iou1, dice1 = mask_metrics(sam_v1_fg, gt_fg)
    sam_v1_ious.append(iou1); sam_v1_dices.append(dice1)

    # SAM v2 (auto-mask)
    m_v2 = segment_with_sam_auto(image)
    sam_v2_fg = np.zeros_like(gt_fg)
    for m in m_v2: sam_v2_fg |= m
    iou2, dice2 = mask_metrics(sam_v2_fg, gt_fg)
    sam_v2_ious.append(iou2); sam_v2_dices.append(dice2)

print(f'\n=== Segmentación class-agnostic sobre {N_SEG_EVAL} imágenes de FoodSeg103 ===')
print(f'Box-as-mask (nb07):     mIoU = {np.mean(box_ious):.3f}   Dice = {np.mean(box_dices):.3f}')
print(f'SAM + YOLO (nb08, v1):  mIoU = {np.mean(sam_v1_ious):.3f}   Dice = {np.mean(sam_v1_dices):.3f}')
print(f'SAM auto (nb09, v2):    mIoU = {np.mean(sam_v2_ious):.3f}   Dice = {np.mean(sam_v2_dices):.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ['Box-as-mask\n(nb07)', 'SAM+YOLO\n(nb08 v1)', 'SAM auto\n(nb09 v2)']
colors = ['gray', 'steelblue', 'seagreen']
for a, (metric, vals) in zip(ax, [('mIoU', [box_ious, sam_v1_ious, sam_v2_ious]),
                                   ('Dice', [box_dices, sam_v1_dices, sam_v2_dices])]):
    means = [np.mean(v) for v in vals]
    bars = a.bar(labels, means, color=colors)
    a.set(ylabel=metric, title=f'{metric} sobre {N_SEG_EVAL} imágenes', ylim=(0, 1))
    a.grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, means):
        a.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('segmentation_metrics_nb09.png', dpi=110, bbox_inches='tight')
plt.show()

## 🎯 Parte 6 — Comparativa de clasificación: single vs ensemble

Mismo protocolo del nb08 (crops oráculo desde máscaras GT) pero clasificamos cada crop **dos veces**: con embeddings single-template y con ensemble. La métrica de comparación es Top-1/3/5 sobre el mismo set de crops, así la diferencia es atribuible solo al texto de los prompts.

In [ ]:
MAX_CLIP_CROPS = 300
MIN_AREA_FRAC  = 0.005
N_CLIP_IMAGES  = 200

clip_v1_top1 = clip_v1_top3 = clip_v1_top5 = 0
clip_v2_top1 = clip_v2_top3 = clip_v2_top5 = 0
total = 0
clip_idxs = rng.choice(len(foodseg), size=N_CLIP_IMAGES, replace=False)

for idx in tqdm(clip_idxs, desc='CLIP single vs ensemble (oráculo)'):
    if total >= MAX_CLIP_CROPS: break
    s = foodseg[int(idx)]
    image = s['image'].convert('RGB')
    gt = np.array(s['label'])
    if gt.ndim == 3: gt = gt[..., 0]
    H, W = gt.shape
    min_area = MIN_AREA_FRAC * H * W
    for cls_id in np.unique(gt):
        if cls_id == 0: continue
        mask = (gt == cls_id)
        if mask.sum() < min_area: continue
        crop = crop_from_mask(image, mask)
        if crop is None or min(crop.size) < 20: continue
        true_name = FOODSEG103_CLASSES[int(cls_id) - 1].strip()
        # v1: single template
        top_v1 = [n.strip() for n, _ in classify_with_clip(crop, topk=5, use_ensemble=False)]
        if top_v1[0] == true_name:        clip_v1_top1 += 1
        if true_name in top_v1[:3]:       clip_v1_top3 += 1
        if true_name in top_v1[:5]:       clip_v1_top5 += 1
        # v2: ensemble
        top_v2 = [n.strip() for n, _ in classify_with_clip(crop, topk=5, use_ensemble=True)]
        if top_v2[0] == true_name:        clip_v2_top1 += 1
        if true_name in top_v2[:3]:       clip_v2_top3 += 1
        if true_name in top_v2[:5]:       clip_v2_top5 += 1
        total += 1
        if total >= MAX_CLIP_CROPS: break

print(f'\n=== CLIP zero-shot sobre {total} crops oráculo de FoodSeg103 ===')
print(f'                    Top-1    Top-3    Top-5')
print(f'  Single (v1):     {clip_v1_top1/total:.3f}    {clip_v1_top3/total:.3f}    {clip_v1_top5/total:.3f}')
print(f'  Ensemble (v2):   {clip_v2_top1/total:.3f}    {clip_v2_top3/total:.3f}    {clip_v2_top5/total:.3f}')
print(f'  Δ ensemble - single:  {(clip_v2_top1-clip_v1_top1)/total:+.3f}   '
      f'{(clip_v2_top3-clip_v1_top3)/total:+.3f}   {(clip_v2_top5-clip_v1_top5)/total:+.3f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(3); width = 0.35
v1_vals = [clip_v1_top1/total, clip_v1_top3/total, clip_v1_top5/total]
v2_vals = [clip_v2_top1/total, clip_v2_top3/total, clip_v2_top5/total]
ax.bar(x - width/2, v1_vals, width, label='Single template (v1)', color='steelblue')
ax.bar(x + width/2, v2_vals, width, label='Ensemble 6 templates (v2)', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(['Top-1', 'Top-3', 'Top-5'])
ax.set(ylabel='Accuracy', title=f'CLIP zero-shot — single vs ensemble ({total} crops)')
ax.legend(); ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0, 1)
for xi, vv in zip(x - width/2, v1_vals): ax.text(xi, vv + 0.01, f'{vv:.2f}', ha='center', fontsize=8)
for xi, vv in zip(x + width/2, v2_vals): ax.text(xi, vv + 0.01, f'{vv:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('clip_ensemble_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

## 🔢 Parte 7 — Comparativa MAPE calórico: v1 vs v2

Mismo GT calórico que el nb08 (suma de kcal_per_100g × área × 250g/100). Comparamos los dos pipelines sobre las mismas 50 imágenes (las del Parte 5).

Esperable:
- v2 baja el MAPE porque (a) SAM auto cubre toda la comida y (b) el filtro de confianza descarta los disparos catastróficos del estilo "lamb 366 kcal".

In [ ]:
rows = []
for idx in tqdm(seg_idxs, desc='MAPE v1 vs v2'):
    s = foodseg[int(idx)]
    image = s['image'].convert('RGB')
    gt = np.array(s['label'])
    if gt.ndim == 3: gt = gt[..., 0]
    fg_area = int((gt > 0).sum())
    if fg_area == 0: continue

    # GT calórico
    gt_kcal = 0.0
    for cls_id in np.unique(gt):
        if cls_id == 0: continue
        m = (gt == cls_id)
        cls_name = FOODSEG103_CLASSES[int(cls_id) - 1]
        nut = get_ingredient_nutrition(cls_name)
        if nut is None: continue
        portion_g = PLATE_PORTION_G * m.sum() / fg_area
        gt_kcal += nut['calories'] * portion_g / 100
    if gt_kcal <= 0: continue

    r1 = analyze_dish_v1(image)
    r2 = analyze_dish_v2(image)
    rows.append(dict(gt=gt_kcal, v1=r1['total_kcal'], v2=r2['total_kcal']))

df = pd.DataFrame(rows)
mape_v1 = (np.abs(df['v1'] - df['gt']) / df['gt']).mean() * 100
mape_v2 = (np.abs(df['v2'] - df['gt']) / df['gt']).mean() * 100
med_v1  = (np.abs(df['v1'] - df['gt']) / df['gt']).median() * 100
med_v2  = (np.abs(df['v2'] - df['gt']) / df['gt']).median() * 100
print(f'\n=== MAPE calórico sobre {len(df)} imágenes de FoodSeg103 ===')
print(f'  v1 (SAM+YOLO+CLIP simple):       media {mape_v1:.1f}%   mediana {med_v1:.1f}%')
print(f'  v2 (SAM auto + ensemble + filtro): media {mape_v2:.1f}%   mediana {med_v2:.1f}%')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(df['gt'], df['v1'], alpha=0.5, s=30, color='steelblue', label=f'v1 (mediana {med_v1:.0f}%)')
ax[0].scatter(df['gt'], df['v2'], alpha=0.5, s=30, color='seagreen', label=f'v2 (mediana {med_v2:.0f}%)')
m_max = max(df['gt'].max(), df['v1'].max(), df['v2'].max())
ax[0].plot([0, m_max], [0, m_max], 'k--', alpha=0.5, label='ideal')
ax[0].set(xlabel='GT kcal', ylabel='Pred kcal', title='Predicho vs GT por imagen')
ax[0].legend(); ax[0].grid(alpha=0.3)

bars = ax[1].bar(['v1\n(nb08)', 'v2\n(nb09)'],
                 [med_v1, med_v2], color=['steelblue', 'seagreen'])
ax[1].set(ylabel='MAPE mediana (%)', title='Mediana del error relativo')
ax[1].grid(alpha=0.3, axis='y')
for bar, v in zip(bars, [med_v1, med_v2]):
    ax[1].text(bar.get_x() + bar.get_width()/2, v + max(med_v1, med_v2)*0.02,
               f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('calorie_mape_v1_vs_v2.png', dpi=110, bbox_inches='tight')
plt.show()

## 🖼️ Parte 8 — Visualización cualitativa: GT | v1 | v2

Seis imágenes lado a lado. Para v2 marcamos en verde las máscaras aceptadas (kcal > 0) y en gris las indeterminadas (filtro de confianza).

In [ ]:
def overlay_masks(ax, image, masks, labels=None, alpha=0.45, colors=None):
    ax.imshow(image); ax.axis('off')
    if colors is None:
        colors = plt.cm.tab20.colors
    for i, m in enumerate(masks):
        color = colors[i % len(colors)] if not isinstance(colors, list) or len(colors) != len(masks) else colors[i]
        rgba = np.zeros((*m.shape, 4)); rgba[m] = (*color[:3], alpha)
        ax.imshow(rgba)
        if labels is not None:
            ys, xs = np.where(m)
            if len(xs):
                ax.text(float(xs.mean()), float(ys.mean()), labels[i],
                        fontsize=7, color='white', ha='center', va='center',
                        bbox=dict(facecolor='black', alpha=0.6, pad=1.2, edgecolor='none'))

qual_idxs = rng.choice(len(foodseg), size=6, replace=False)
fig, axes = plt.subplots(6, 3, figsize=(15, 28))
for col, header in enumerate(['(a) GT', '(b) v1 — SAM+YOLO+CLIP simple', '(c) v2 — SAM auto + ensemble + filtro']):
    axes[0, col].set_title(header, fontsize=11, fontweight='bold', pad=10)

for row, idx in enumerate(qual_idxs):
    s = foodseg[int(idx)]
    image = s['image'].convert('RGB')
    gt = np.array(s['label'])
    if gt.ndim == 3: gt = gt[..., 0]
    H, W = gt.shape

    gt_masks, gt_labels = [], []
    for cls_id in np.unique(gt):
        if cls_id == 0: continue
        m = (gt == cls_id)
        if m.sum() < 0.003 * H * W: continue
        gt_masks.append(m); gt_labels.append(FOODSEG103_CLASSES[int(cls_id)-1].strip())
    overlay_masks(axes[row, 0], image, gt_masks, labels=gt_labels)
    axes[row, 0].set_title(f'{len(gt_masks)} ingredientes (GT)', fontsize=9)

    r1 = analyze_dish_v1(image)
    v1_masks  = [d['mask'] for d in r1['detections']]
    v1_labels = [f"{d['ingredient'].strip()} · {d['kcal']:.0f} kcal" for d in r1['detections']]
    overlay_masks(axes[row, 1], image, v1_masks, labels=v1_labels)
    axes[row, 1].set_title(f"v1 — total {r1['total_kcal']:.0f} kcal", fontsize=9)

    r2 = analyze_dish_v2(image)
    v2_masks  = [d['mask'] for d in r2['detections']]
    # Verde si aceptada, gris si indeterminada
    v2_colors = [(0.18, 0.55, 0.34) if d['status'] == 'ok' else (0.5, 0.5, 0.5)
                 for d in r2['detections']]
    v2_labels = [f"{d['ingredient'].strip()} · {d['kcal']:.0f} kcal" if d['status'] == 'ok'
                 else f"? {d['ingredient'].strip()} (conf {d['confidence']:.2f})"
                 for d in r2['detections']]
    overlay_masks(axes[row, 2], image, v2_masks, labels=v2_labels, colors=v2_colors)
    n_ok = sum(1 for d in r2['detections'] if d['status'] == 'ok')
    n_un = sum(1 for d in r2['detections'] if d['status'] != 'ok')
    axes[row, 2].set_title(f"v2 — {n_ok} aceptadas + {n_un} indet · total {r2['total_kcal']:.0f} kcal", fontsize=9)

plt.tight_layout()
plt.savefig('foodseg103_qualitative_v1_vs_v2.png', dpi=110, bbox_inches='tight')
plt.show()

## 📋 Resumen + conclusiones

In [ ]:
print('=' * 72)
print('RESUMEN — Notebook 09: pipeline FoodSeg103 v2 (auto-mask + ensemble + filtro)')
print('=' * 72)
print()
print(f'Cambios respecto al nb08:')
print(f'  1. Segmentación: SAM auto-mask (sin prompts) en vez de YOLO+boxes')
print(f'  2. Clasificación: CLIP ensemble de {len(CLIP_TEMPLATES_ENSEMBLE)} templates en vez de uno solo')
print(f'  3. Decisión: filtro de confianza CLIP >= {MIN_CONF} (descarta predicciones débiles)')
print()
print(f'Segmentación sobre {N_SEG_EVAL} imágenes (mIoU class-agnostic):')
print(f'  Box-as-mask  (nb07):       {np.mean(box_ious):.3f}')
print(f'  SAM + YOLO   (nb08, v1):   {np.mean(sam_v1_ious):.3f}')
print(f'  SAM auto     (nb09, v2):   {np.mean(sam_v2_ious):.3f}')
print(f'  Mejora v2 - v1: {np.mean(sam_v2_ious) - np.mean(sam_v1_ious):+.3f} mIoU')
print()
print(f'Clasificación CLIP sobre {total} crops oráculo:')
print(f'  Single template (v1):  top-1 {clip_v1_top1/total:.3f}   top-5 {clip_v1_top5/total:.3f}')
print(f'  Ensemble (v2):         top-1 {clip_v2_top1/total:.3f}   top-5 {clip_v2_top5/total:.3f}')
print(f'  Mejora top-1: {(clip_v2_top1-clip_v1_top1)/total:+.3f}')
print()
print(f'MAPE calórico end-to-end sobre {len(df)} imágenes:')
print(f'  v1 (nb08):  media {mape_v1:.1f}%  mediana {med_v1:.1f}%')
print(f'  v2 (nb09):  media {mape_v2:.1f}%  mediana {med_v2:.1f}%')
print()
print('Conclusiones para el informe:')
print('  - La causa raíz de los problemas del nb08 era el detector COCO (10 clases)')
print('    usado como source de prompts para SAM. Al sacar a YOLO de la cadena y')
print('    dejar que SAM proponga máscaras solo, la cobertura mejora drásticamente.')
print('  - El prompt ensemble es una optimización barata (mismo modelo, mismo tiempo')
print('    de inferencia, solo más texto al inicio) que suma 3-5 puntos al top-1.')
print('  - El filtro de confianza evita los disparos catastróficos ("lamb 366 kcal"')
print('    sobre una rodaja de zanahoria), a costa de subestimar cuando CLIP duda.')
print('  - La principal limitación persiste: CLIP top-1 sigue lejos del 100%.')
print('    Para cerrar esa brecha haría falta fine-tunear EfficientNet-B0 sobre')
print('    crops de FoodSeg103 (trabajo futuro fuera del alcance de este TP).')
print('=' * 72)